# 00 · Colab runner: open this every session and click Runtime → Run all

It runs the next unfinished experiments from `configs/pipeline.yaml` and stops before the free-tier
GPU budget runs out. Everything is saved to Google Drive (`MyDrive/govi-ai/`), so:

* if Colab disconnects, just open this notebook again and **Run all**: finished steps are skipped and
  training resumes from its last epoch;
* you don't need to watch it. Logs land in `govi-ai/logs/` and sync to your PC.

**First:** Runtime → Change runtime type → **T4 GPU**. If no GPU is available today, CPU-only steps still run.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
GOVI = '/content/drive/MyDrive/govi-ai'
DATA = '/content/data'
SESSION_HOURS = 3.5   # free tier: keep under the daily GPU allowance
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU: only CPU steps will run'

In [ ]:
!git clone -q https://github.com/Kalana-Lakshan/govi-isuru-ai-service.git /content/govi-isuru-ai-service 2>/dev/null || git -C /content/govi-isuru-ai-service pull -q
%cd /content/govi-isuru-ai-service
!pip install -q -e ".[train]"
!git log --oneline -1

In [ ]:
import os
if not os.path.exists(f'{DATA}/manifest.csv'):   # local disk is wiped every session
    !python scripts/pack_dataset.py --unpack --out $GOVI/dataset --data-root $DATA
!python scripts/pipeline.py --govi $GOVI --status

In [ ]:
!python scripts/pipeline.py --data-root $DATA --govi $GOVI --hours $SESSION_HOURS

In [ ]:
!python scripts/pipeline.py --govi $GOVI --status